# NYC Yellow Taxi · версия 2
## Временная кросс-валидация, бизнес-ошибка и приложение Streamlit

**Задача:** прогноз числа очищенных поездок по пяти боро Нью-Йорка на следующий местный час.
Все метрики и выводы рассчитываются при запуске; результаты версии 1 сюда не подставлены.

### Запуск
1. Выберите CPU в Google Colab и выполните ячейки по порядку.
2. Проверьте параметры: штрафы 3/1 — сценарные; основной критерий можно переключить с MAE на бизнес-ошибку.
3. Новый тест по умолчанию — январь 2025. Декабрь 2024 уже изучен и не считается независимым.
4. Скачайте итоговый ZIP. Он содержит Streamlit, Docker, исходники, модель, результаты и отчёт.
5. Для просмотра Streamlit в Colab включите переключатель в последнем разделе.

Исходники проекта встроены в ноутбук: отдельная загрузка ZIP не требуется.
Обучение идёт в изолированном Python-окружении, поэтому библиотеки Colab не понижаются.
Точное окружение сохраняется после установки. Интернет необходим для зависимостей и файлов TLC.
Первые 12–13 файлов скачиваются и агрегируются последовательно небольшими пакетами.

**Статус поставки:** код и Streamlit проверяются на искусственных тестовых данных;
новые научные результаты появятся только после полного запуска на TLC. Docker и публикация
репозитория — отдельные этапы проверки. Подробности — в файле VALIDATION.md внутри проекта.

In [ ]:
from pathlib import Path
import base64, io, zipfile, sys, os, subprocess, json, csv, html
from IPython.display import display, Markdown, HTML, Image

PROJECT = (Path('/content') if Path('/content').exists() else Path.cwd()) / 'nyc_taxi_v2'
PROJECT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for entry in archive.infolist():
        target = (PROJECT / entry.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError('Некорректный путь в архиве исходников')
    archive.extractall(PROJECT)
print('Проект распакован:', PROJECT)
display(Markdown((PROJECT / 'VALIDATION.md').read_text(encoding='utf-8')))

## 1. Конфигурация и бизнес-сценарий

Стоимость недооценки 3 и переоценки 1 — **заданные сценарные штрафы**, не измеренные затраты автопарка.
Business Loss = Σ max(0, факт − прогноз) × 3 + Σ max(0, прогноз − факт) × 1.
Если выбирать по BusinessLossPerHour, LightGBM использует квантиль 0.75; по MAE — L1 loss.
Среднее на час нужно для сопоставления месяцев разной длины, суммарная стоимость также сохраняется.

Конкретная стоимость ошибки зависит от юнит-экономики автопарка; модель демонстрирует возможность
оптимизации под асимметричные штрафы (undersupply penalization).

Задержка 1 час — исходное допущение ретроспективной задачи. Для реального применения требуется
измерить доступность очищенных поездок, потому что стоимость и дистанция известны после завершения поездки.
Можно провести отдельный эксперимент с задержкой 2–24 часа, не называя её фактической без данных.

In [ ]:
YEAR = 2024
INCLUDE_NEW_TEST = True  # Январь 2025; False оставляет только уже изученный декабрь.
SELECTION_METRIC = 'MAE'  # Или 'BusinessLossPerHour'
COST_UNDERESTIMATION = 3.0
COST_OVERESTIMATION = 1.0
AVAILABILITY_DELAY_HOURS = 1
OPTUNA_TRIALS_PER_FAMILY = 8
BUSINESS_USE = 'Не согласовано'
BUSINESS_MAX_MAE = None  # Не придумываем допустимый порог.

config = json.loads((PROJECT / 'config.json').read_text(encoding='utf-8'))
config.update(year=YEAR, include_new_test=INCLUDE_NEW_TEST, selection_metric=SELECTION_METRIC,
              cost_underestimation=COST_UNDERESTIMATION, cost_overestimation=COST_OVERESTIMATION,
              availability_delay_hours=AVAILABILITY_DELAY_HOURS, trials=OPTUNA_TRIALS_PER_FAMILY,
              business_use=BUSINESS_USE, business_max_mae=BUSINESS_MAX_MAE)
(PROJECT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
display(config)

## 2. Изолированное окружение

Эта ячейка не выполняет `%pip install` в системную среду Colab. Создаётся отдельный venv,
в котором запускаются обучение, тесты и Streamlit. В notebook остаётся только отображение файлов.
`requirements.txt` задаёт совместимые диапазоны; точные установленные версии сохраняются
в `requirements.lock.txt`. Для воспроизведения конкретного запуска используйте его с той же версией Python.
Если установка или `pip check` завершается ошибкой, выполнение останавливается, а не скрывает конфликт.

In [ ]:
ENVIRONMENT = PROJECT / '.venv'
PYTHON = ENVIRONMENT / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
if not PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENVIRONMENT)], check=True)

def run(args):
    env = os.environ.copy()
    env.update(PYTHONUTF8='1', PYTHONUNBUFFERED='1', MPLBACKEND='Agg')
    with subprocess.Popen([str(PYTHON)] + list(args), cwd=PROJECT, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace') as process:
        for line in process.stdout:
            print(line, end='')
        if process.wait() != 0:
            raise RuntimeError('Команда завершилась ошибкой. См. вывод выше.')

dependency_file = 'requirements.lock.txt' if (PROJECT / 'requirements.lock.txt').exists() else 'requirements.txt'
run(['-m', 'pip', 'install', '--no-cache-dir', '--prefer-binary', '-r', dependency_file])
run(['-m', 'pip', 'check'])
lock = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], cwd=PROJECT, text=True)
(PROJECT / 'requirements.lock.txt').write_text(lock, encoding='utf-8')
run(['-c', 'import platform; print("Python:", platform.python_version())'])

## 3. Автотесты до работы с данными

Проверяется временная причинность признаков, задержки 1/2/24 часа, очистка,
инвалидация кэша, пропуски/DST, неправильный и устаревший вход, сериализация,
полная цепочка на искусственных данных и интерфейс Streamlit.
Эти тесты не измеряют качество модели на TLC и не создают демонстрационные метрики в отчёте.

In [ ]:
run(['-m', 'pytest', '-q', 'tests'])

## 4. Данные и контроль качества

Источник: [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
Четыре столбца читаются пакетами. Оставляем даты месяца файла, конечные значения,
0 < расстояние ≤ 100 миль, 0 ≤ стоимость ≤ $500 и зоны пяти боро.
Эти правила заданы проектом; исключённые записи не обязательно ошибки.
Кэш связан с параметрами очистки, справочником и хешем агрегата.
Неожиданное отсутствие целого часа в сыром источнике вызывает ошибку, а не превращается в ноль спроса.
DST-часы становятся неизвестными; цели и окна с неизвестной историей исключаются.

In [ ]:
run(['-m', 'taxi_project.cli', 'prepare'])

In [ ]:
ART = PROJECT / 'artifacts'

def show_csv(name, limit=50):
    with (ART / name).open(encoding='utf-8') as stream:
        rows = list(csv.reader(stream))
    head = ''.join('<th>' + html.escape(v) + '</th>' for v in rows[0])
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(v) + '</td>' for v in row) + '</tr>'
                   for row in rows[1:limit+1])
    display(HTML('<div style="overflow:auto"><table><thead><tr>' + head + '</tr></thead><tbody>'
                 + body + '</tbody></table></div>'))

show_csv('data_audit.csv')

## 5. EDA на обучающем периоде

Распределение таргета, квантили, временные профили, автокорреляции, спектр и кандидаты аномалий.
Причины пиков не приписываются погоде или событиям без внешних данных.
В отчёте автоматически появятся конкретные выводы текущего запуска.

In [ ]:
run(['-m', 'taxi_project.cli', 'eda'])
for name in ['eda.png', 'seasonality.png']:
    display(Image(filename=str(ART / name)))
show_csv('target_statistics.csv')

## 6. CV, подбор, отбор признаков и финальная проверка

| Этап | Период / правило |
|---|---|
| Внешние фолды CV | Август, сентябрь, октябрь 2024 |
| Внутренняя ранняя остановка | Дни с −28 по −14 перед внешним месяцем |
| Внутренняя калибровка | Последние 14 дней перед внешним месяцем |
| Подбор | Optuna для Ridge и LightGBM, одинаковые фолды |
| Признаки | Календарь → +лаги → +окна; сравнение и выбор |
| Правило выбора | В пределах 1% лучшего среднего: меньше признаков, меньше std, быстрее прогноз |
| Финальное обучение | Январь–октябрь; параметры зафиксированы по CV |
| Финальная калибровка | Ноябрь |
| Историческая оценка | Декабрь, уже изученный в версии 1 |
| Новый holdout | Январь следующего года при включённом параметре |

Календарные фолды сохраняют продолжительность несмотря на исключения DST.
Скейлер и кодирование обучаются только внутри train. Внешний месяц не используется для early stopping.
CV используется для выбора: её оценка не является независимым доказательством качества.
После просмотра нового holdout нельзя менять модель и продолжать называть его независимым.

In [ ]:
run(['-m', 'taxi_project.cli', 'train'])

In [ ]:
display(Markdown('### Сравнение по CV и отбор признаков'))
show_csv('cv_summary.csv')
display(Image(filename=str(ART / 'cv_comparison.png')))
display(Markdown('### Калибровка: сравнение способов на CV'))
show_csv('interval_cv_summary.csv')
display(Markdown('### Декабрь и новый holdout: без изменения выбора'))
show_csv('evaluation.csv')

## 7. Ошибки, интерпретация и мониторинг

Интервалы сравниваем по фактическому покрытию и ширине, а не только по надписи «90%».
Два инструмента объяснения LightGBM: gain и sklearn permutation importance; дополнительно
сохранены групповые перестановки, коэффициенты и перестановочная важность Ridge.
Это диагностика связей, не причинное доказательство. Объяснение не используется для повторного подбора.
PSI и предупреждения качества видны также в Streamlit; технические пороги явно отделены от бизнес-SLA.

In [ ]:
for name in ['correlations.png', 'importance.png', 'forecast_December_seen.png',
             'errors_December_seen.png', 'forecast_January_new.png', 'errors_January_new.png']:
    if (ART / name).exists():
        display(Image(filename=str(ART / name)))
show_csv('errors_is_holiday.csv')
display(json.loads((ART / 'monitoring.json').read_text(encoding='utf-8')))
display(Markdown((ART / 'REPORT.md').read_text(encoding='utf-8')))

## 8. Архив проекта и результатов

Архив содержит приложение, модель, тесты, Docker и точный список установленных библиотек.
Сырые поездки, кэш и виртуальная среда не включаются. Скачайте архив до завершения сеанса Colab.
Для GitHub публикуйте исходники и README; крупные артефакты и секреты не добавляйте.

In [ ]:
import shutil
shutil.copy2(PROJECT / 'requirements.lock.txt', ART / 'requirements.lock.txt')
ARCHIVE = PROJECT.parent / 'NYC_Taxi_v2_results.zip'
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in PROJECT.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(PROJECT)
        if any(part in ['.venv', '__pycache__', '.pytest_cache', 'raw', 'cache', '.git'] for part in rel.parts):
            continue
        bundle.write(path, str(Path('nyc_taxi_v2') / rel))
print('Архив:', ARCHIVE)
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except ImportError:
    print('В локальной среде скачайте архив по указанному пути.')

## 9. Streamlit в Colab — опциональный просмотр

Установите `START_STREAMLIT = True`, чтобы открыть приложение ниже. Используется встроенный
iframe-прокси Colab без внешнего публичного туннеля. Этот просмотр зависит от правил браузера
и Colab; локальный запуск из ZIP и Docker описаны в README.
Документация прокси: [Google Colab output utilities](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).

In [ ]:
START_STREAMLIT = False
if START_STREAMLIT:
    import time, urllib.request
    if 'streamlit_process' not in globals() or streamlit_process.poll() is not None:
        streamlit_log = (PROJECT / 'streamlit.log').open('w', encoding='utf-8')
        streamlit_process = subprocess.Popen([str(PYTHON), '-m', 'streamlit', 'run', 'app.py',
            '--server.port=8501', '--server.address=0.0.0.0', '--server.headless=true'],
            cwd=PROJECT, stdout=streamlit_log, stderr=subprocess.STDOUT)
    ready = False
    for _ in range(30):
        try:
            with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=1) as response:
                ready = response.status == 200
            if ready:
                break
        except OSError:
            time.sleep(1)
    if not ready:
        raise RuntimeError('Streamlit не запустился. См. streamlit.log в проекте.')
    try:
        from google.colab.output import serve_kernel_port_as_iframe
        serve_kernel_port_as_iframe(8501, height=1000)
    except ImportError:
        print('Откройте http://localhost:8501 в локальном браузере.')

## 10. Готовность и ограничения

Код реализует доработки, но завершённость научного результата определяется реальными таблицами
текущего запуска. Нельзя заранее обещать победу бустинга или 90%-покрытие.
Публикация на GitHub, проверка Docker и подтверждение онлайн-доступности признаков — отдельные
пункты готовности. Число машин, фактические деньги и бизнес-SLA без данных заказчика не рассчитываются.
Стратификация классов, F1 и confusion matrix здесь неприменимы: задача регрессионная и временная.

In [ ]:
display(Markdown((PROJECT / 'ACCEPTANCE.md').read_text(encoding='utf-8')))

## Приложение: читаемые исходники

Далее приведён тот же код, который распакован в проект и выполняется в отдельном окружении.
Для изменений редактируйте файлы в папке проекта. После изменения протокола создайте отдельный
запуск и не выдавайте просмотренный holdout за новый.

### taxi_project/core.py

<details><summary>Показать исходный код</summary>

```python
"""Shared data contract, causal features, model artifact and monitoring."""
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error

SCHEMA_VERSION = 2


@dataclass(frozen=True)
class Config:
    year: int = 2024
    availability_delay_hours: int = 1
    max_distance: float = 100.0
    max_amount: float = 500.0
    batch_size: int = 200000
    seed: int = 42
    trials: int = 8
    n_estimators: int = 1500
    alpha: float = .1
    include_new_test: bool = True
    business_use: str = 'Не согласовано'
    business_max_mae: float | None = None
    cost_underestimation: float = 3.0
    cost_overestimation: float = 1.0
    selection_metric: str = 'MAE'

    def __post_init__(self):
        if not 1 <= self.availability_delay_hours <= 24:
            raise ValueError('Задержка доступности должна быть от 1 до 24 часов')
        if not 0 < self.alpha < 1:
            raise ValueError('alpha должна лежать между 0 и 1')
        if self.trials < 1 or self.n_estimators < 1:
            raise ValueError('Число испытаний и деревьев должно быть положительным')
        if min(self.cost_underestimation, self.cost_overestimation) <= 0:
            raise ValueError('Сценарные штрафы должны быть положительными')
        if self.selection_metric not in ['MAE', 'BusinessLossPerHour']:
            raise ValueError('Неизвестная метрика выбора')


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def clean_signature(config, zone_ids, lookup_sha):
    payload = dict(schema=SCHEMA_VERSION, rule='city-month-finite-v2',
                   max_distance=config.max_distance, max_amount=config.max_amount,
                   zone_ids=sorted(int(z) for z in zone_ids), lookup_sha=lookup_sha)
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def validate_panel(frame, allow_nan=False):
    if set(frame.columns) != {'pickup_hour', 'trips_count'}:
        raise ValueError('Ожидаются только pickup_hour и trips_count')
    out = frame.copy()
    out['pickup_hour'] = pd.to_datetime(out.pickup_hour, errors='raise')
    if out.empty or out.pickup_hour.isna().any():
        raise ValueError('Пустые данные или даты')
    if out.pickup_hour.dt.tz is not None:
        raise ValueError('Нужны местные часы Нью-Йорка без timezone')
    if not out.pickup_hour.eq(out.pickup_hour.dt.floor('h')).all():
        raise ValueError('Время должно совпадать с началом часа')
    if out.pickup_hour.duplicated().any():
        raise ValueError('Дублирующиеся часы')
    out['trips_count'] = pd.to_numeric(out.trips_count, errors='raise')
    known = out.trips_count.dropna()
    if not np.isfinite(known).all() or (known < 0).any():
        raise ValueError('Число поездок должно быть конечным и неотрицательным')
    if not allow_nan and out.trips_count.isna().any():
        raise ValueError('Пропуски в числе поездок')
    out = out.sort_values('pickup_hour').reset_index(drop=True)
    expected = pd.date_range(out.pickup_hour.min(), out.pickup_hour.max(), freq='h')
    if len(out) != len(expected):
        raise ValueError('Пропущены часы в сетке')
    return out


def features(frame, config):
    out = validate_panel(frame, allow_nan=True)
    dt = out.pickup_hour.dt
    out['hour'] = dt.hour
    out['weekday'] = dt.dayofweek
    out['is_weekend'] = (dt.dayofweek >= 5).astype(int)
    # Federal observed holidays, defined explicitly; no external calendar dependency.
    dates = USFederalHolidayCalendar().holidays(out.pickup_hour.min().normalize(),
                                               out.pickup_hour.max().normalize())
    out['is_holiday'] = out.pickup_hour.dt.normalize().isin(dates).astype(int)
    for name, period in [('hour', 24), ('weekday', 7)]:
        out[name + '_sin'] = np.sin(2 * np.pi * out[name] / period)
        out[name + '_cos'] = np.cos(2 * np.pi * out[name] / period)
    delay = config.availability_delay_hours
    for lag in sorted({delay, delay + 1, delay + 2, 24, 168}):
        out[f'lag_{lag}'] = out.trips_count.shift(lag)
    for window in (3, 24, 168):
        prior = out.trips_count.shift(delay).rolling(window, min_periods=window)
        out[f'roll_mean_{window}'] = prior.mean()
        out[f'roll_std_{window}'] = prior.std(ddof=0)
    return out


def feature_sets(config):
    calendar = ['hour', 'weekday', 'is_weekend', 'is_holiday',
                'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos']
    lags = [f'lag_{v}' for v in sorted({config.availability_delay_hours,
            config.availability_delay_hours + 1, config.availability_delay_hours + 2, 24, 168})]
    rolling = [f'roll_{stat}_{w}' for w in (3, 24, 168) for stat in ('mean', 'std')]
    return {'calendar': calendar, 'lags': calendar + lags, 'full': calendar + lags + rolling}


def metrics(y, prediction):
    y, prediction = np.asarray(y, float), np.asarray(prediction, float)
    if y.shape != prediction.shape or not y.size or not np.isfinite(y).all() or not np.isfinite(prediction).all():
        raise ValueError('Некорректные массивы метрик')
    return dict(MAE=float(mean_absolute_error(y, prediction)),
                RMSE=float(np.sqrt(mean_squared_error(y, prediction))),
                WAPE_pct=float(100 * np.abs(y - prediction).sum() / y.sum()) if y.sum() else float('nan'))


def business_metrics(y, prediction, config):
    error = np.asarray(y, float) - np.asarray(prediction, float)
    cost = config.cost_underestimation * np.maximum(0, error) + config.cost_overestimation * np.maximum(0, -error)
    return {'BusinessLoss': float(cost.sum()), 'BusinessLossPerHour': float(cost.mean())}


def radius(y, prediction, method, alpha=.1):
    error = np.abs(np.asarray(y) - np.asarray(prediction))
    scale = np.sqrt(np.maximum(1, prediction)) if method == 'scaled' else np.ones(len(error))
    scores = np.sort(error / scale)
    rank = math.ceil((len(scores) + 1) * (1 - alpha))
    if len(scores) < 30 or rank > len(scores):
        raise ValueError('Недостаточно данных для калибровки')
    return float(scores[rank - 1])


def bounds(prediction, q, method):
    prediction = np.asarray(prediction)
    scale = np.sqrt(np.maximum(1, prediction)) if method == 'scaled' else np.ones(len(prediction))
    return np.maximum(0, prediction - q * scale), prediction + q * scale


@dataclass
class ForecastModel:
    pipeline: object
    name: str
    columns: list
    config: Config
    interval_method: str
    interval_q: float
    fitted_until: str
    reference: dict
    schema_version: int = SCHEMA_VERSION

    def predict_features(self, table):
        if self.name == 'WeeklyNaive':
            return table.lag_168.to_numpy()
        return np.maximum(0, self.pipeline.predict(table[self.columns]))

    def forecast(self, history, target_time, live=False, now=None):
        h = validate_panel(history)
        target = pd.Timestamp(target_time)
        if target.tzinfo is not None or pd.isna(target) or target != target.floor('h'):
            raise ValueError('Момент прогноза должен быть началом локального часа')
        last = target - pd.Timedelta(hours=self.config.availability_delay_hours)
        if h.pickup_hour.max() != last:
            raise ValueError('История устарела или содержит ещё недоступные часы')
        if live:
            current = pd.Timestamp.now(tz='America/New_York') if now is None else pd.Timestamp(now)
            if current.tzinfo is None:
                raise ValueError('Для live-проверки now должен содержать timezone')
            current = current.tz_convert('America/New_York').tz_localize(None).floor('h')
            if target != current:
                raise ValueError('Прогноз не соответствует текущему часу Нью-Йорка')
        grid = pd.date_range(last - pd.Timedelta(hours=167), target, freq='h')
        if grid.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT').isna().any():
            raise ValueError('История затрагивает неоднозначные часы DST')
        tail = h[h.pickup_hour >= grid.min()]
        if len(tail) != 168:
            raise ValueError('Нужны минимум 168 полных часов истории')
        future = pd.DataFrame({'pickup_hour': pd.date_range(last + pd.Timedelta(hours=1), target, freq='h'),
                               'trips_count': np.nan})
        row = features(pd.concat([tail, future], ignore_index=True), self.config).tail(1)
        if row[self.columns].isna().any().any() or pd.isna(row.lag_168.iloc[0]):
            raise ValueError('Недостаточно истории для признаков')
        pred = self.predict_features(row)
        lo, hi = bounds(pred, self.interval_q, self.interval_method)
        return pd.DataFrame({'pickup_hour': [target], 'prediction': pred, 'lower': lo, 'upper': hi})

    def save(self, path):
        joblib.dump(self, path)


def load_model(path):
    # Load only trusted local artifacts, never arbitrary uploaded pickle/joblib files.
    obj = joblib.load(path)
    if not isinstance(obj, ForecastModel) or obj.schema_version != SCHEMA_VERSION:
        raise ValueError('Несовместимая версия артефакта')
    return obj


def reference_distribution(values):
    values = np.asarray(values, float)
    cuts = np.unique(np.quantile(values, np.linspace(0, 1, 11)))
    edges = np.r_[-np.inf, cuts[1:-1], np.inf]
    counts = np.histogram(values, bins=edges)[0] + .5
    return {'edges': edges.tolist(), 'probabilities': (counts / counts.sum()).tolist()}


def monitor(values, reference, actual=None, prediction=None, coverage=None, max_mae=None):
    values = np.asarray(values, float)
    if not len(values) or not np.isfinite(values).all():
        raise ValueError('Мониторинг требует конечные наблюдения')
    counts = np.histogram(values, bins=reference['edges'])[0] + .5
    observed = counts / counts.sum()
    expected = np.asarray(reference['probabilities'])
    psi = float(np.sum((observed - expected) * np.log(observed / expected)))
    alerts = []
    if psi > .2:
        alerts.append('PSI > 0.2: проверить изменение распределения (технический ориентир, не бизнес-SLA)')
    output = {'psi': psi, 'n': len(values), 'alerts': alerts}
    if actual is not None:
        output.update(metrics(actual, prediction))
        if max_mae is not None and output['MAE'] > max_mae:
            alerts.append('MAE выше согласованного бизнес-порога')
    if coverage is not None:
        output['coverage'] = float(coverage)
        if coverage < .85:
            alerts.append('Покрытие <85% при номинале 90%: проверить интервалы; порог исследовательский')
    return output

```

</details>

### taxi_project/data.py

<details><summary>Показать исходный код</summary>

```python
"""Streaming city aggregation with content-checked, configuration-aware cache."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from .core import digest, clean_signature

BASE = 'https://d37ci6vzurychx.cloudfront.net'
COLUMNS = ['tpep_pickup_datetime', 'PULocationID', 'trip_distance', 'total_amount']


def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size:
        return destination
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=Retry(total=4, backoff_factor=1,
                  status_forcelist=[429, 500, 502, 503, 504])))
    temporary = destination.with_suffix(destination.suffix + '.part')
    try:
        with session.get(url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with temporary.open('wb') as stream:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        stream.write(chunk)
        if not temporary.stat().st_size:
            raise ValueError('Пустой ответ источника')
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
        session.close()
    return destination


def cache_read(path, signature):
    path = Path(path)
    meta_path = path.with_suffix('.json')
    if not path.exists() or not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta['signature'] != signature or meta['aggregate_sha256'] != digest(path):
            return None
        table = pd.read_parquet(path)
        if (table.trips_count.sum() != meta['kept_rows'] or table.pickup_hour.duplicated().any()
                or meta['raw_rows'] != sum(meta[k] for k in
                    ['kept_rows', 'rejected_date', 'rejected_values', 'rejected_zone'])):
            return None
        return table, meta
    except (ValueError, KeyError, OSError):
        return None


def aggregate_file(raw_path, year, month, config, zone_ids):
    low = pd.Timestamp(year, month, 1)
    high = low + pd.offsets.MonthBegin(1)
    counts = None
    raw_hours = set()
    audit = dict(year=year, month=month, raw_rows=0, rejected_date=0,
                 rejected_values=0, rejected_zone=0, kept_rows=0)
    with pq.ParquetFile(raw_path) as parquet:
        if not set(COLUMNS).issubset(parquet.schema_arrow.names):
            raise ValueError('Неполная схема исходного Parquet')
        for batch in parquet.iter_batches(batch_size=config.batch_size, columns=COLUMNS):
            frame = batch.to_pandas()
            audit['raw_rows'] += len(frame)
            time = pd.to_datetime(frame.tpep_pickup_datetime, errors='coerce')
            in_month = time.ge(low) & time.lt(high)
            raw_hours.update(time[in_month].dt.floor('h').unique())
            distance = pd.to_numeric(frame.trip_distance, errors='coerce')
            amount = pd.to_numeric(frame.total_amount, errors='coerce')
            finite = np.isfinite(distance) & np.isfinite(amount)
            valid = finite & distance.gt(0) & distance.le(config.max_distance) & amount.ge(0) & amount.le(config.max_amount)
            zones = frame.PULocationID.isin(zone_ids)
            audit['rejected_date'] += int((~in_month).sum())
            audit['rejected_values'] += int((in_month & ~valid).sum())
            audit['rejected_zone'] += int((in_month & valid & ~zones).sum())
            keep = in_month & valid & zones
            grouped = time[keep].dt.floor('h').value_counts(sort=False)
            counts = grouped if counts is None else counts.add(grouped, fill_value=0)
            audit['kept_rows'] += int(keep.sum())
    if counts is None or audit['kept_rows'] == 0:
        raise ValueError('Месяц не содержит допустимых поездок')
    hours = pd.date_range(low, high, freq='h', inclusive='left')
    dst = hours.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT').isna()
    missing = hours.difference(pd.DatetimeIndex(list(raw_hours)))
    unexpected = missing.difference(hours[dst])
    if len(unexpected):
        raise ValueError(f'В исходнике нет ни одной записи за часы: {unexpected[:5].tolist()}')
    result = counts.rename_axis('pickup_hour').rename('trips_count').reset_index()
    result['trips_count'] = result.trips_count.astype('int64')
    result = result.sort_values('pickup_hour').reset_index(drop=True)
    assert result.trips_count.sum() == audit['kept_rows']
    return result, audit


def prepare(root, config):
    root = Path(root)
    raw, cache, artifacts = [root / name for name in ['raw', 'cache', 'artifacts']]
    for path in (raw, cache, artifacts):
        path.mkdir(parents=True, exist_ok=True)
    lookup_path = download(f'{BASE}/misc/taxi_zone_lookup.csv', cache / 'taxi_zone_lookup.csv')
    lookup = pd.read_csv(lookup_path)
    zone_ids = set(lookup.loc[lookup.Borough.isin(['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']), 'LocationID'])
    signature = clean_signature(config, zone_ids, digest(lookup_path))
    months = [(config.year, month) for month in range(1, 13)]
    if config.include_new_test:
        months.append((config.year + 1, 1))
    frames, audits = [], []
    for year, month in months:
        output = cache / f'city_{year}_{month:02d}_{signature[:12]}.parquet'
        cached = cache_read(output, signature)
        if cached is None:
            url = f'{BASE}/trip-data/yellow_tripdata_{year}-{month:02d}.parquet'
            path = download(url, raw / url.rsplit('/', 1)[-1])
            frame, audit = aggregate_file(path, year, month, config, zone_ids)
            audit.update(signature=signature, source_url=url, source_sha256=digest(path))
            temporary = output.with_suffix('.tmp.parquet')
            frame.to_parquet(temporary, index=False)
            temporary.replace(output)
            audit['aggregate_sha256'] = digest(output)
            meta = output.with_suffix('.json')
            tmp_meta = meta.with_suffix('.tmp')
            tmp_meta.write_text(json.dumps(audit, indent=2), encoding='utf-8')
            tmp_meta.replace(meta)
            path.unlink()
        else:
            frame, audit = cached
        frames.append(frame)
        audits.append(audit)
        print(f'{year}-{month:02d}: {audit["kept_rows"]:,} поездок', flush=True)
    sparse = pd.concat(frames, ignore_index=True)
    end = pd.Timestamp(config.year + 1, 2 if config.include_new_test else 1, 1)
    grid = pd.date_range(pd.Timestamp(config.year, 1, 1), end, freq='h', inclusive='left')
    panel = sparse.set_index('pickup_hour').reindex(grid, fill_value=0).rename_axis('pickup_hour').reset_index()
    assert panel.trips_count.sum() == sparse.trips_count.sum()
    bad = grid.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT').isna()
    panel.loc[bad, 'trips_count'] = np.nan
    panel.to_parquet(artifacts / 'panel.parquet', index=False)
    pd.DataFrame(audits).to_csv(artifacts / 'data_audit.csv', index=False)
    return panel

```

</details>

### taxi_project/experiment.py

<details><summary>Показать исходный код</summary>

```python
"""Calendar CV, Optuna selection, ablation, interval comparison and final evaluation."""
from dataclasses import asdict
from pathlib import Path
from time import perf_counter, process_time
import json
import pickle
import threading
import platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import psutil
import optuna
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance
from .core import (ForecastModel, features, feature_sets, metrics, business_metrics,
                   radius, bounds, reference_distribution, monitor, digest)


def scores(y, pred, config):
    return {**metrics(y, pred), **business_metrics(y, pred, config)}


def calendar_folds(data, year):
    for month in [8, 9, 10]:
        start = pd.Timestamp(year, month, 1)
        end = start + pd.offsets.MonthBegin(1)
        train = data[data.pickup_hour < start]
        valid = data[(data.pickup_hour >= start) & (data.pickup_hour < end)]
        if train.empty or valid.empty:
            raise ValueError('Не хватает данных для календарной CV')
        assert train.pickup_hour.max() < valid.pickup_hour.min()
        yield f'{year}-{month:02d}', start, train, valid


def make_pipeline(family, params, columns, config):
    categorical = [c for c in ['hour', 'weekday'] if c in columns]
    numeric = [c for c in columns if c not in categorical]
    if family == 'Ridge':
        prep = ColumnTransformer([('calendar', OneHotEncoder(handle_unknown='ignore'), categorical),
                                  ('numeric', StandardScaler(), numeric)])
        estimator = Ridge(alpha=params['alpha'], solver='lsqr')
    else:
        prep = ColumnTransformer([('numeric', 'passthrough', columns)], remainder='drop')
        prep.set_output(transform='pandas')
        kwargs = dict(objective='regression_l1', learning_rate=.05, n_estimators=config.n_estimators,
                      n_jobs=2, random_state=config.seed, deterministic=True, force_col_wise=True,
                      verbosity=-1)
        if config.selection_metric == 'BusinessLossPerHour':
            kwargs.update(objective='quantile', alpha=config.cost_underestimation /
                          (config.cost_underestimation + config.cost_overestimation))
        kwargs.update(params)
        estimator = lgb.LGBMRegressor(**kwargs)
    return Pipeline([('preprocess', prep), ('model', estimator)])


def measured_fit(pipeline, x, y, **kwargs):
    process = psutil.Process()
    baseline = process.memory_info().rss
    samples = [baseline]
    stop = threading.Event()
    def sample():
        while not stop.wait(.02):
            samples.append(process.memory_info().rss)
    thread = threading.Thread(target=sample, daemon=True)
    thread.start()
    start, cpu = perf_counter(), process_time()
    try:
        pipeline.fit(x, y, **kwargs)
    finally:
        stop.set()
        thread.join()
    elapsed = perf_counter() - start
    return dict(fit_seconds=elapsed, cpu_seconds=process_time() - cpu,
                process_peak_rss_mb=max(samples + [process.memory_info().rss]) / 2**20,
                additional_rss_mb=max(0, max(samples) - baseline) / 2**20)


def fit_fold(family, params, columns, train, boundary, config):
    # Dedicated internal early-stop period AND separate calibration period.
    calibration_start = boundary - pd.Timedelta(days=14)
    stop_start = boundary - pd.Timedelta(days=28)
    embargo = pd.Timedelta(hours=config.availability_delay_hours - 1)
    fit = train[train.pickup_hour < calibration_start - embargo]
    calibration = train[train.pickup_hour >= calibration_start]
    if min(len(fit), len(calibration)) < 30:
        raise ValueError('Недостаточно данных внутренних окон')
    if family == 'WeeklyNaive':
        return None, fit, calibration, 0, dict(fit_seconds=0, cpu_seconds=0, process_peak_rss_mb=0, additional_rss_mb=0)
    count = 0
    if family == 'LightGBM':
        core = train[train.pickup_hour < stop_start - embargo]
        early = train[(train.pickup_hour >= stop_start) & (train.pickup_hour < calibration_start)]
        initial = make_pipeline(family, params, columns, config)
        initial.named_steps['preprocess'].fit(core[columns])
        x_early = initial.named_steps['preprocess'].transform(early[columns])
        initial.fit(core[columns], core.trips_count,
                    model__eval_set=[(x_early, early.trips_count)],
                    model__callbacks=[lgb.early_stopping(60, first_metric_only=True, verbose=False)])
        count = max(1, initial.named_steps['model'].best_iteration_)
        params = {**params, 'n_estimators': count}
    pipeline = make_pipeline(family, params, columns, config)
    timing = measured_fit(pipeline, fit[columns], fit.trips_count)
    return pipeline, fit, calibration, count, timing


def pred(pipeline, family, frame, columns):
    if family == 'WeeklyNaive':
        return frame.lag_168.to_numpy()
    return np.maximum(0, pipeline.predict(frame[columns]))


def evaluate_candidate(data, family, params, columns, config, label, capture=False):
    rows, interval_rows, held_predictions = [], [], []
    for fold, boundary, train, valid in calendar_folds(data, config.year):
        start = perf_counter()
        pipeline, fit, calibration, count, timing = fit_fold(family, params, columns, train, boundary, config)
        train_pred = pred(pipeline, family, fit, columns)
        cal_pred = pred(pipeline, family, calibration, columns)
        elapsed = []
        for _ in range(5):
            started = perf_counter()
            valid_pred = pred(pipeline, family, valid, columns)
            elapsed.append(perf_counter() - started)
        row = dict(candidate=label, family=family, fold=fold, n_features=len(columns),
                   n_train=len(fit), n_validation=len(valid), best_iteration=count,
                   train_MAE=metrics(fit.trips_count, train_pred)['MAE'],
                   inference_median_s=float(np.median(elapsed)), inference_p95_s=float(np.quantile(elapsed,.95)),
                   model_bytes=len(pickle.dumps(pipeline)) if pipeline is not None else 0,
                   total_fold_seconds=perf_counter()-start, **timing, **scores(valid.trips_count, valid_pred, config))
        rows.append(row)
        if capture:
            output = valid[['pickup_hour','trips_count']].copy()
            output['prediction'] = valid_pred
            output['fold'] = fold
            held_predictions.append(output)
            for method in ['absolute', 'scaled']:
                q = radius(calibration.trips_count, cal_pred, method, config.alpha)
                lower, upper = bounds(valid_pred, q, method)
                interval_rows.append(dict(candidate=label, fold=fold, method=method,
                    coverage=float(((valid.trips_count >= lower) & (valid.trips_count <= upper)).mean()),
                    mean_width=float(np.mean(upper-lower)), n=len(valid)))
    return pd.DataFrame(rows), pd.DataFrame(interval_rows), held_predictions


def summary(rows):
    return rows.groupby(['candidate','family'], as_index=False).agg(
        MAE_mean=('MAE','mean'), MAE_std=('MAE','std'), RMSE_mean=('RMSE','mean'),
        WAPE_mean=('WAPE_pct','mean'), BusinessLossPerHour_mean=('BusinessLossPerHour','mean'),
        BusinessLossPerHour_std=('BusinessLossPerHour','std'), train_MAE_mean=('train_MAE','mean'),
        fit_seconds_mean=('fit_seconds','mean'), inference_median_s=('inference_median_s','median'),
        peak_rss_mb=('process_peak_rss_mb','max'), model_bytes=('model_bytes','max'),
        n_features=('n_features','first'))


def run_experiment(panel, config, artifacts):
    artifacts = Path(artifacts)
    artifacts.mkdir(parents=True, exist_ok=True)
    sets = feature_sets(config)
    table = features(panel, config).dropna().reset_index(drop=True)
    tune_data = table[table.pickup_hour < pd.Timestamp(config.year,11,1)]
    configs = {'WeeklyNaive': ('WeeklyNaive', {}, sets['full'])}
    all_rows = []
    first, _, _ = evaluate_candidate(tune_data, 'WeeklyNaive', {}, sets['full'], config, 'WeeklyNaive')
    all_rows.append(first)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    for family in ['Ridge','LightGBM']:
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=config.seed))
        def objective(trial):
            if family == 'Ridge':
                params = {'alpha':trial.suggest_float('alpha', .1, 1000, log=True)}
            else:
                params = dict(num_leaves=trial.suggest_int('num_leaves',15,63),
                    min_child_samples=trial.suggest_int('min_child_samples',30,200),
                    reg_lambda=trial.suggest_float('reg_lambda',.1,10,log=True))
            rows, _, _ = evaluate_candidate(tune_data,family,params,sets['full'],config,f'{family}_trial{trial.number}')
            for column in ['MAE','BusinessLossPerHour']:
                trial.set_user_attr(column+'_std', float(rows[column].std()))
            return float(rows[config.selection_metric].mean())
        study.optimize(objective, n_trials=config.trials)
        study.trials_dataframe().to_csv(artifacts/f'optuna_{family}.csv',index=False)
        best = study.best_params
        for subset, columns in sets.items():
            label = f'{family}_{subset}'
            configs[label] = (family,best,columns)
            rows, _, _ = evaluate_candidate(tune_data,family,best,columns,config,label)
            all_rows.append(rows)
            print(f'CV {label}: {config.selection_metric}={rows[config.selection_metric].mean():.3f}', flush=True)
    cv = pd.concat(all_rows,ignore_index=True)
    cv.to_csv(artifacts/'cv_folds.csv',index=False)
    leaderboard = summary(cv)
    metric = config.selection_metric+'_mean'
    best_score = leaderboard[metric].min()
    # Predeclared simplicity rule: within 1% of best, prefer fewer features, lower variability, then latency.
    eligible = leaderboard[leaderboard[metric] <= best_score*1.01 + 1e-12]
    chosen = eligible.sort_values(['n_features',config.selection_metric+'_std','inference_median_s','candidate']).iloc[0].candidate
    leaderboard['selected'] = leaderboard.candidate.eq(chosen)
    leaderboard.to_csv(artifacts/'cv_summary.csv',index=False)
    family, params, columns = configs[chosen]
    _, intervals, oof = evaluate_candidate(tune_data,family,params,columns,config,chosen,capture=True)
    intervals.to_csv(artifacts/'interval_cv.csv',index=False)
    interval_summary = intervals.groupby('method',as_index=False).agg(coverage=('coverage','mean'),mean_width=('mean_width','mean'))
    interval_summary['coverage_gap'] = abs(interval_summary.coverage-(1-config.alpha))
    method = interval_summary.sort_values(['coverage_gap','mean_width','method']).iloc[0].method
    interval_summary.to_csv(artifacts/'interval_cv_summary.csv',index=False)
    (artifacts/'selection.json').write_text(json.dumps({'candidate':chosen,'family':family,
        'params':params,'features':columns,'interval_method':method,'selection_metric':config.selection_metric,
        'note':'Fixed using August–October CV before evaluating December/January'},indent=2),encoding='utf-8')
    pd.concat(oof).to_csv(artifacts/'cv_predictions.csv',index=False)
    fitted_until = pd.Timestamp(config.year,11,1)
    fit = table[table.pickup_hour < fitted_until-pd.Timedelta(hours=config.availability_delay_hours-1)]
    calibration = table[(table.pickup_hour >= fitted_until) & (table.pickup_hour < pd.Timestamp(config.year,12,1))]
    selected_by_family = {'WeeklyNaive':'WeeklyNaive'}
    for f in ['Ridge','LightGBM']:
        subset = leaderboard[leaderboard.family.eq(f)].sort_values(metric)
        selected_by_family[f] = chosen if family == f else subset.iloc[0].candidate
    models, test_rows, predictions = {}, [], []
    for f, label in selected_by_family.items():
        _, parameters, cols = configs[label]
        if f == 'WeeklyNaive':
            pipeline = None
            timing = dict(fit_seconds=0,cpu_seconds=0,process_peak_rss_mb=0,additional_rss_mb=0)
        else:
            if f == 'LightGBM':
                rounds = int(np.median(cv[cv.candidate.eq(label)].best_iteration))
                parameters = {**parameters,'n_estimators':max(1,rounds)}
            pipeline = make_pipeline(f,parameters,cols,config)
            timing = measured_fit(pipeline,fit[cols],fit.trips_count)
        cal_pred = pred(pipeline,f,calibration,cols)
        q = radius(calibration.trips_count,cal_pred,method,config.alpha)
        model = ForecastModel(pipeline,f,cols,config,method,q,str(fitted_until),reference_distribution(fit.trips_count))
        models[label] = model
        for period, begin, end in [('December_seen',pd.Timestamp(config.year,12,1),pd.Timestamp(config.year+1,1,1)),
                                    ('January_new',pd.Timestamp(config.year+1,1,1),pd.Timestamp(config.year+1,2,1))]:
            subset = table[(table.pickup_hour>=begin)&(table.pickup_hour<end)]
            if subset.empty:
                continue
            timings=[]
            for _ in range(5):
                start=perf_counter(); p=model.predict_features(subset); timings.append(perf_counter()-start)
            lower, upper = bounds(p,q,method)
            coverage=float(subset.trips_count.between(lower,upper).mean())
            test_rows.append(dict(period=period,candidate=label,selected=label==chosen,n=len(subset),
                coverage=coverage,mean_width=float(np.mean(upper-lower)),inference_median_s=float(np.median(timings)),
                inference_p95_s=float(np.quantile(timings,.95)),model_bytes=len(pickle.dumps(model)),
                **timing,**scores(subset.trips_count,p,config)))
            if label==chosen:
                output=subset[['pickup_hour','trips_count','hour','weekday','is_holiday']].copy()
                output['prediction'],output['lower'],output['upper']=p,lower,upper
                output['period']=period
                predictions.append(output)
    champion=models[chosen]
    champion.save(artifacts/'model.joblib')
    results=pd.DataFrame(test_rows)
    results.to_csv(artifacts/'evaluation.csv',index=False)
    predictions=pd.concat(predictions,ignore_index=True)
    predictions['abs_error']=abs(predictions.trips_count-predictions.prediction)
    predictions['covered']=predictions.trips_count.between(predictions.lower,predictions.upper)
    predictions.to_csv(artifacts/'predictions.csv',index=False)
    for dimension in ['hour','weekday','is_holiday']:
        predictions.groupby(['period',dimension]).agg(MAE=('abs_error','mean'),coverage=('covered','mean'),n=('covered','size')).to_csv(artifacts/f'errors_{dimension}.csv')
    # Importance belongs to development data; do not use January for feature selection.
    explain_models(models,selected_by_family,calibration,artifacts,config)
    monitors=[]
    for period, group in predictions.groupby('period'):
        monitors.append({'period':period,**monitor(group.trips_count,champion.reference,
            group.trips_count,group.prediction,group.covered.mean(),config.business_max_mae)})
    (artifacts/'monitoring.json').write_text(json.dumps(monitors,ensure_ascii=False,indent=2),encoding='utf-8')
    (artifacts/'config.json').write_text(json.dumps(asdict(config),ensure_ascii=False,indent=2),encoding='utf-8')
    versions={p:version(p) for p in ['numpy','pandas','pyarrow','scikit-learn','lightgbm','optuna','streamlit']}
    versions.update(python=platform.python_version(),cpu_logical=psutil.cpu_count(),model_threads=2,
                    gpu_used=False,model_sha256=digest(artifacts/'model.joblib'))
    (artifacts/'environment.json').write_text(json.dumps(versions,indent=2),encoding='utf-8')
    return table,leaderboard,results,champion


def explain_models(models, selected_by_family, calibration, artifacts, config):
    tree=models[selected_by_family['LightGBM']]
    booster=tree.pipeline.named_steps['model'].booster_
    pd.Series(booster.feature_importance(importance_type='gain'),index=tree.columns,name='gain').to_csv(artifacts/'importance_gain.csv')
    # Second implementation: sklearn permutation importance, diagnostic only.
    subset=calibration.iloc[::max(1,len(calibration)//400)]
    result=permutation_importance(tree.pipeline,subset[tree.columns],subset.trips_count,
                                  scoring='neg_mean_absolute_error',n_repeats=5,random_state=config.seed,n_jobs=1)
    pd.DataFrame({'feature':tree.columns,'mae_increase':result.importances_mean,
                  'std':result.importances_std}).to_csv(artifacts/'importance_permutation.csv',index=False)
    # Blocked joint permutation preserves correlation within lag/rolling groups (diagnostic caveats remain).
    rows=[]
    base=metrics(subset.trips_count,tree.predict_features(subset))['MAE']
    rng=np.random.default_rng(config.seed)
    for group,prefixes in [('lags',('lag_',)),('rolling',('roll_',)),('calendar',('hour','weekday','is_'))]:
        cols=[c for c in tree.columns if c.startswith(prefixes)]
        if not cols:
            continue
        deltas=[]
        for _ in range(5):
            shifted=subset.copy()
            order=np.roll(np.arange(len(subset)),int(rng.integers(24,max(25,len(subset)-24))))
            shifted.loc[:,cols]=subset[cols].to_numpy()[order]
            deltas.append(metrics(subset.trips_count,tree.predict_features(shifted))['MAE']-base)
        rows.append({'group':group,'mae_increase':np.mean(deltas),'std':np.std(deltas)})
    pd.DataFrame(rows).to_csv(artifacts/'importance_groups.csv',index=False)
    ridge=models[selected_by_family['Ridge']]
    names=ridge.pipeline.named_steps['preprocess'].get_feature_names_out()
    pd.DataFrame({'feature':names,'coefficient':ridge.pipeline.named_steps['model'].coef_}).to_csv(
        artifacts/'ridge_coefficients.csv',index=False)
    ridge_perm=permutation_importance(ridge.pipeline,subset[ridge.columns],subset.trips_count,
        scoring='neg_mean_absolute_error',n_repeats=5,random_state=config.seed,n_jobs=1)
    pd.DataFrame({'feature':ridge.columns,'mae_increase':ridge_perm.importances_mean,
                  'std':ridge_perm.importances_std}).to_csv(artifacts/'ridge_permutation.csv',index=False)

```

</details>

### taxi_project/reporting.py

<details><summary>Показать исходный код</summary>

```python
"""Figures and evidence-based Russian reports; no pre-filled experimental results."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import periodogram


def save(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def eda(panel, config, artifacts):
    artifacts=Path(artifacts)
    train=panel[panel.pickup_hour < pd.Timestamp(config.year,10,1)].copy()
    train['hour']=train.pickup_hour.dt.hour
    train['weekday']=train.pickup_hour.dt.dayofweek
    values=train.trips_count.dropna()
    stats=values.describe(percentiles=[.01,.05,.5,.95,.99])
    stats.to_csv(artifacts/'target_statistics.csv',header=['value'])
    center=values.median()
    mad=(values-center).abs().median()
    outliers=train[abs(train.trips_count-center)>6*max(1,mad)].copy()
    outliers.to_csv(artifacts/'eda_extreme_hours.csv',index=False)
    fig,axes=plt.subplots(2,2,figsize=(14,8))
    train.set_index('pickup_hour').trips_count.resample('D').sum(min_count=1).plot(ax=axes[0,0])
    axes[0,0].set(title='Train: поездки по дням',xlabel='Дата',ylabel='Поездки за сутки')
    axes[0,1].hist(values,bins=50)
    axes[0,1].set(title='Train: распределение целевой переменной',xlabel='Поездки за час',ylabel='Количество часов')
    train.groupby('hour').trips_count.mean().plot.bar(ax=axes[1,0])
    axes[1,0].set(title='Профиль суток',xlabel='Час Нью-Йорка',ylabel='Среднее поездок/час')
    train.groupby('weekday').trips_count.mean().plot.bar(ax=axes[1,1])
    axes[1,1].set(title='Профиль недели',xlabel='День недели: 0 — понедельник',ylabel='Среднее поездок/час')
    save(fig,artifacts/'eda.png')
    corr={str(lag):float(train.trips_count.autocorr(lag)) for lag in [1,24,168]}
    # Interpolation is only for this spectrum, never for model features/targets.
    spectrum=train.trips_count.interpolate(limit_direction='both')
    freq,power=periodogram(spectrum,detrend='linear')
    period=1/freq[1:]
    keep=(period>=2)&(period<=400)
    fig,ax=plt.subplots(figsize=(12,4))
    ax.plot(period[keep],power[1:][keep])
    ax.axvline(24,color='orange',label='24 часа');ax.axvline(168,color='red',label='168 часов')
    ax.set(title='Train: периодограмма',xlabel='Период, часы',ylabel='Спектральная мощность')
    ax.legend();save(fig,artifacts/'seasonality.png')
    peak=int(train.groupby('hour').trips_count.mean().idxmax())
    trough=int(train.groupby('hour').trips_count.mean().idxmin())
    text=f'''# EDA: выводы по обучающему периоду

Медиана: {center:.2f} поездки/час; 95-й процентиль: {values.quantile(.95):.2f}.
Доля нулей среди известных часов: {values.eq(0).mean():.2%}.
Пропуски: {train.trips_count.isna().sum()} часов; их не заменяем будущими значениями.
Наибольший средний объём в {peak:02d}:00, наименьший — в {trough:02d}:00.
Автокорреляции: 1 час {corr['1']:.3f}; 24 часа {corr['24']:.3f}; 168 часов {corr['168']:.3f}.
Это поддерживает гипотезы суточной и недельной сезонности; полезность признаков оценивается отдельно на CV.

За пределами медианы ±6 MAD найдено {len(outliers)} часов. Это диагностический порог,
не основание удалять высокий реальный спрос. Часы выгружены для проверки причин;
погода и события отсутствуют, поэтому объяснения не приписываем.
Правила очистки исходных поездок проверяются отдельным аудитом; исключённые записи не обязательно ошибочны.
Спектр использует интерполяцию только для визуализации. Годовая сезонность на одном году не доказана.
'''
    (artifacts/'EDA_REPORT.md').write_text(text,encoding='utf-8')


def final_report(table, leaderboard, results, model, config, artifacts):
    artifacts=Path(artifacts)
    if not config.include_new_test:
        for name in ['forecast_January_new.png','errors_January_new.png']:
            (artifacts/name).unlink(missing_ok=True)
    chosen=leaderboard[leaderboard.selected].iloc[0].candidate
    predictions=pd.read_csv(artifacts/'predictions.csv',parse_dates=['pickup_hour'])
    cv=pd.read_csv(artifacts/'cv_folds.csv')
    numeric=table[table.pickup_hour<pd.Timestamp(config.year,10,1)].select_dtypes('number')
    correlation=numeric.corr()
    correlation.to_csv(artifacts/'feature_correlations.csv')
    fig,ax=plt.subplots(figsize=(11,9))
    im=ax.imshow(correlation,vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(len(correlation)),correlation.columns,rotation=90,fontsize=7)
    ax.set_yticks(range(len(correlation)),correlation.columns,fontsize=7)
    ax.set_title('Train: корреляции таргета и признаков');fig.colorbar(im,ax=ax,label='Pearson r')
    save(fig,artifacts/'correlations.png')
    fig,ax=plt.subplots(figsize=(13,5))
    names=leaderboard.candidate.tolist()
    positions=np.arange(len(names))
    ax.bar(positions,leaderboard.MAE_mean,yerr=leaderboard.MAE_std,capsize=4,label='Validation: среднее ± std')
    ax.plot(positions,leaderboard.train_MAE_mean,'o-',color='orange',label='Train MAE')
    ax.set_xticks(positions,names,rotation=25,ha='right')
    ax.set(title='Календарная CV: качество и признаки',xlabel='Модель / набор признаков',ylabel='MAE, поездки/час')
    ax.legend();save(fig,artifacts/'cv_comparison.png')
    for period,group in predictions.groupby('period'):
        zoom=group.head(24*14)
        fig,axes=plt.subplots(2,1,figsize=(14,8))
        axes[0].plot(zoom.pickup_hour,zoom.trips_count,label='Факт')
        axes[0].plot(zoom.pickup_hour,zoom.prediction,label='Прогноз')
        axes[0].fill_between(zoom.pickup_hour,zoom.lower,zoom.upper,alpha=.2,label='Интервал: номинал 90%')
        axes[0].set(title=f'{period}: Нью-Йорк; покрытие месяца {group.covered.mean():.1%}',
                    xlabel='Локальное время Нью-Йорка',ylabel='Поездки за час')
        axes[0].legend()
        axes[1].hist(group.trips_count-group.prediction,bins=50)
        axes[1].set(title='Распределение остатков',xlabel='Факт минус прогноз, поездки',ylabel='Количество часов')
        save(fig,artifacts/f'forecast_{period}.png')
        fig,axes=plt.subplots(1,2,figsize=(13,4))
        group.groupby('hour').abs_error.mean().plot.bar(ax=axes[0])
        axes[0].set(title='Ошибки по часам',xlabel='Час Нью-Йорка',ylabel='MAE, поездки/час')
        group.groupby('weekday').abs_error.mean().plot.bar(ax=axes[1])
        axes[1].set(title='Ошибки по дням недели',xlabel='0 — понедельник',ylabel='MAE, поездки/час')
        save(fig,artifacts/f'errors_{period}.png')
    gain=pd.read_csv(artifacts/'importance_gain.csv',index_col=0).iloc[:,0]
    permutation=pd.read_csv(artifacts/'importance_permutation.csv').set_index('feature')
    fig,axes=plt.subplots(1,2,figsize=(14,7))
    gain.nlargest(12).sort_values().plot.barh(ax=axes[0])
    axes[0].set(title='LightGBM: gain',xlabel='Суммарное уменьшение потерь',ylabel='Признак')
    permutation.mae_increase.nlargest(12).sort_values().plot.barh(ax=axes[1])
    axes[1].set(title='sklearn: перестановочная важность',xlabel='Рост MAE при перестановке',ylabel='Признак')
    save(fig,artifacts/'importance.png')
    paragraphs=[]
    for period,group in results.groupby('period'):
        selected=group[group.selected].iloc[0]
        baseline=group[group.candidate.eq('WeeklyNaive')].iloc[0]
        improvement=100*(1-selected.MAE/baseline.MAE) if baseline.MAE else np.nan
        paragraphs.append(f'''### {period}

MAE выбранной модели {selected.MAE:.2f}; RMSE {selected.RMSE:.2f}; WAPE {selected.WAPE_pct:.2f}%.
Снижение MAE относительно WeeklyNaive {improvement:.2f}%. Отрицательное число означает ухудшение.
Сценарная Business Loss {selected.BusinessLoss:.2f}, на час {selected.BusinessLossPerHour:.2f}.
Покрытие интервала {selected.coverage:.2%}, номинал {1-config.alpha:.0%}; средняя ширина {selected.mean_width:.2f}.
{'Номинальное покрытие на этом периоде не достигнуто.' if selected.coverage < 1-config.alpha else 'Эмпирическое покрытие достигло номинала на этом периоде; это не гарантия будущего покрытия.'}
''')
    best=leaderboard[leaderboard.selected].iloc[0]
    lag_gain=gain.idxmax() if len(gain) else 'нет'
    ablation=[]
    for family in ['Ridge','LightGBM']:
        rows=leaderboard[leaderboard.family.eq(family)].set_index('candidate')
        full=rows.loc[f'{family}_full','MAE_mean']
        for group in ['calendar','lags']:
            value=rows.loc[f'{family}_{group}','MAE_mean']
            ablation.append(f'- {family}: {group} MAE {value:.2f}, полный набор {full:.2f}; '
                            f'разница сокращённый минус полный {value-full:+.2f} поездки/час.')
    ablation_text='\n'.join(ablation)
    report=f'''# NYC Taxi v2 — результаты выполнения

## Задача и бизнес-интерпретация
Суммарное число очищенных поездок Yellow Taxi по пяти боро Нью-Йорка на следующий локальный час.
Назначение: {config.business_use}. Если заказчик ещё не определил действие по прогнозу, готовность бизнес-сценария не подтверждена.
Сценарные штрафы: недооценка {config.cost_underestimation}, переоценка {config.cost_overestimation} за поездку.
Business Loss = сумма положительных недооценок × первый штраф + сумма переоценок × второй штраф.
Конкретная стоимость ошибки зависит от юнит-экономики автопарка; модель демонстрирует возможность
оптимизации под асимметричные штрафы (undersupply penalization).
Если трактовать штрафы как доллары, результат — условный сценарный долларовый ущерб, не измеренные потери.
Нет перевода поездок в число машин без длительности поездок, загрузки и времени доступности автомобилей.

## Протокол
Optuna: {config.trials} испытаний на каждое семейство; внешние месяцы CV — август, сентябрь, октябрь {config.year}.
Внутри каждого фолда последние 14 дней — калибровка; предыдущие 14 — early stopping.
После early stopping модель переобучается на данных до калибровки, внешний месяц не участвует в обучении.
Для каждого семейства проверены календарь, календарь+лаги, полный набор с окнами.
Выбор по {config.selection_metric}: в пределах 1% от лучшего среднего предпочитаем меньше признаков,
затем меньший разброс и время. Это заранее заданное исследовательское правило, не бизнес-SLA.
CV использована для выбора, её метрики оптимистичны относительно независимой проверки.
Финальное обучение — до ноября с учётом задержки, калибровка — ноябрь.
Декабрь {config.year} уже изучен в версии 1; January_new — новый holdout только при первом зафиксированном запуске.
Повторный подбор по January_new лишает его независимости.

## Выбранная модель и признаки
Выбрано {chosen}: {len(model.columns)} признаков.
CV MAE {best.MAE_mean:.2f} ± {best.MAE_std:.2f}; train MAE {best.train_MAE_mean:.2f}.
Разрыв train/validation — повод для анализа сложности, не автоматическое доказательство переобучения.
Таблица cv_summary.csv содержит все группы, времена, размер модели и память.
{ablation_text}
Положительная разница означает преимущество полного набора по MAE на CV, отрицательная — сокращённого.
Если выбран бизнес-критерий, окончательный выбор может отличаться от лучшего MAE.
Наиболее высокий gain у признака {lag_gain}; gain не показывает причинное влияние и знак эффекта.
Permutation importance измеряет изменение MAE; коррелирующие лаги могут делить важность,
а перестановка создаёт нереалистичные сочетания. Групповая циклическая перестановка сохранена отдельно.
Объяснение на ноябре не используется для пересмотра зафиксированного выбора.
Для Ridge отдельно сохранены коэффициенты после предобработки и перестановочная важность.
Для WeeklyNaive интерпретация непосредственная: прогноз равен значению 168 часов назад.

{''.join(paragraphs)}

## Ресурсы и мониторинг
Инференс измерен 5 раз: медиана и p95 относятся к пакету строк, не API-SLA.
Память — выборочный peak RSS всего процесса, не точная память только модели; дополнительный RSS оценён приближённо.
Время CV fit отражает финальное обучение фолда; total_fold_seconds включает внутреннюю раннюю остановку и оценку.
Размер — сериализованный объект. LightGBM использует 2 CPU-потока, GPU не используется.
monitoring.json содержит PSI, качество и предупреждения. PSI >0.2 и покрытие <85% — диагностические
ориентиры, не согласованные бизнес-пороги; изменение сезонного состава само может менять распределение.
Контроль дрейфа не доказывает причину ухудшения. Для действующего сервиса нужен плановый запуск мониторинга.

## Ограничения
Прогнозируем реализованные очищенные поездки, не отклонённые заказы и не весь спрос.
Задержка истории в эксперименте: {config.availability_delay_hours} ч. Это параметр сценария, не измеренная задержка TLC.
Стоимость/дистанция известны после завершения поездки, поэтому готовность лагов онлайн не подтверждена.
Федеральные праздники — календарь pandas с переносом выходных; городские события не представлены.
DST-часы и затронутые окна исключаются; годовая устойчивость не доказана.
Ни улучшение качества, ни покрытие интервалов не гарантируются заранее.
Для внедрения нужны подтверждённый поток данных, бизнес-KPI и эксплуатационная проверка.

## Файлы
cv_folds.csv, cv_summary.csv — сравнение и ablation; optuna_*.csv — подбор;
interval_cv*.csv — выбор интервалов; evaluation.csv — декабрь и новый тест;
predictions.csv — прогнозы; errors_*.csv — ошибки сегментов;
importance_*.csv — интерпретация; environment.json — версии и хеш модели;
model.joblib — доверенный локальный артефакт; data_audit.csv — происхождение и очистка.
'''
    (artifacts/'REPORT.md').write_text(report,encoding='utf-8')

```

</details>

### app.py

<details><summary>Показать исходный код</summary>

```python
"""Streamlit interface for verified artifacts, diagnostics and next-hour forecasts."""
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import streamlit as st
from taxi_project.core import load_model, business_metrics, Config

st.set_page_config(page_title='NYC Taxi — спрос на следующий час',page_icon='🚕',layout='wide')
st.title('🚕 Поездки Yellow Taxi по Нью-Йорку')
st.caption('Исследовательская модель • прогноз на один час • пять боро • локальное время Нью-Йорка')
root=Path(os.environ.get('TAXI_ARTIFACTS',str(Path(__file__).parent/'artifacts')))
if (root/'run_status.json').exists():
    try:
        status=json.loads((root/'run_status.json').read_text(encoding='utf-8'))
        if status.get('status')!='complete':
            st.warning('Текущий запуск ещё не завершён. Дождитесь успешного окончания обучения и отчёта.')
            st.stop()
    except (ValueError,OSError):
        st.error('Некорректный статус запуска. Повторите обучение и формирование отчёта.')
        st.stop()
if not (root/'evaluation.csv').exists():
    st.info('Сначала выполните ноутбук Colab и распакуйте его архив результатов. Поместите папку artifacts рядом с app.py.')
    st.markdown('Демонстрационные метрики не подставляются. После обучения здесь появятся сравнение моделей, графики и прогноз.')
    st.stop()

try:
    results=pd.read_csv(root/'evaluation.csv')
    predictions=pd.read_csv(root/'predictions.csv',parse_dates=['pickup_hour'])
    cv=pd.read_csv(root/'cv_summary.csv')
    config=json.loads((root/'config.json').read_text(encoding='utf-8'))
except (OSError,ValueError,KeyError) as exc:
    st.error(f'Не удалось прочитать комплект результатов: {exc}')
    st.stop()

period=st.sidebar.selectbox('Период оценки',results.period.unique().tolist())
if period=='December_seen':
    st.sidebar.warning('Декабрь уже изучен в версии 1. Это историческая проверка.')
else:
    st.sidebar.info('Независимость января сохраняется только до изменения модели по его результатам.')
st.sidebar.markdown('### Сценарные штрафы')
under=st.sidebar.number_input('Недооценка одной поездки',min_value=.01,value=float(config['cost_underestimation']))
over=st.sidebar.number_input('Переоценка одной поездки',min_value=.01,value=float(config['cost_overestimation']))
st.sidebar.caption('Значения 3 и 1 — условный сценарий. Это не измеренные финансовые потери. Изменение полей пересчитывает только бизнес-ошибку выбранной модели.')
selected=results[results.period.eq(period)&results.selected.eq(True)].iloc[0]
group=predictions[predictions.period.eq(period)]
scenario=Config(**{**config,'cost_underestimation':under,'cost_overestimation':over})
cost=business_metrics(group.trips_count,group.prediction,scenario)
a,b,c,d=st.columns(4)
a.metric('MAE, поездки/час',f'{selected.MAE:.1f}')
b.metric('WAPE',f'{selected.WAPE_pct:.2f}%')
c.metric('Покрытие интервала',f'{selected.coverage:.1%}')
d.metric('Сценарный штраф / час',f'{cost["BusinessLossPerHour"]:.1f}')
if selected.coverage<.9:
    st.warning('Номинальное покрытие 90% на этом периоде не достигнуто.')
tabs=st.tabs(['Прогнозы и ошибки','Сравнение и признаки','Следующий час','Мониторинг и отчёт'])
with tabs[0]:
    st.subheader('Факт и прогноз')
    days=sorted(group.pickup_hour.dt.date.unique())
    start=st.selectbox('Начало графика',days)
    length=st.slider('Показать дней',1,31,7)
    subset=group[(group.pickup_hour>=pd.Timestamp(start)) &
                 (group.pickup_hour<pd.Timestamp(start)+pd.Timedelta(days=length))]
    chart=subset.set_index('pickup_hour')[['trips_count','prediction','lower','upper']]
    st.line_chart(chart.rename(columns={'trips_count':'Факт','prediction':'Прогноз','lower':'Нижняя граница','upper':'Верхняя граница'}))
    st.caption('Ось X — локальное время; ось Y — поездки за час. Границы — эмпирический прогнозный интервал.')
    st.subheader('Ошибки по часам')
    st.bar_chart(group.groupby('hour').abs_error.mean().rename('MAE, поездки/час'))
    st.dataframe(group.nlargest(15,'abs_error')[['pickup_hour','trips_count','prediction','abs_error']])
    st.download_button('Скачать прогнозы CSV',group.to_csv(index=False).encode('utf-8-sig'),file_name=f'{period}_predictions.csv')
with tabs[1]:
    st.subheader('Три календарных фолда: среднее и разброс')
    st.dataframe(cv)
    st.caption('Эти данные использованы при выборе. Числа нового теста не входят в подбор.')
    st.subheader('Зафиксированные модели: проверка периода')
    st.dataframe(results[results.period.eq(period)])
    st.caption('BusinessLoss в таблице использует штрафы конфигурации обучения; боковая панель меняет только карточку сценария.')
    for name in ['cv_comparison.png','importance.png']:
        if (root/name).exists():
            st.image(str(root/name))
    st.caption('Gain и перестановочная важность не являются причинным влиянием признаков.')
with tabs[2]:
    st.subheader('Прогноз следующего часа')
    st.info('Артефакт загружается только из локальной папки результатов. Не используйте чужие непроверенные model.joblib.')
    st.caption('Очищенный предыдущий час может быть недоступен в реальном времени. Пока задержка не подтверждена источником, это исследовательский сценарий.')
    uploaded=st.file_uploader('История CSV: pickup_hour, trips_count; минимум 168 полных часов',type=['csv'])
    try:
        if uploaded is not None:
            history=pd.read_csv(uploaded,parse_dates=['pickup_hour'])
        elif (root/'panel.parquet').exists():
            history=pd.read_parquet(root/'panel.parquet').tail(168).copy()
        else:
            history=None
        if history is not None:
            default=pd.Timestamp(history.pickup_hour.max())+pd.Timedelta(hours=config['availability_delay_hours'])
            target=st.text_input('Начало целевого часа, местное время',value=str(default))
            live=st.checkbox('Проверить соответствие текущему часу Нью-Йорка',value=False)
            if st.button('Рассчитать прогноз'):
                model=load_model(root/'model.joblib')
                forecast=model.forecast(history,target,live=live)
                st.dataframe(forecast)
                st.download_button('Скачать прогноз',forecast.to_csv(index=False).encode(),file_name='next_hour.csv')
    except (ValueError,KeyError,OSError,TypeError) as exc:
        st.error(f'Прогноз не выполнен: {exc}')
with tabs[3]:
    st.subheader('Диагностика данных и качества')
    if (root/'monitoring.json').exists():
        monitoring=json.loads((root/'monitoring.json').read_text(encoding='utf-8'))
        for item in monitoring:
            if item['period']==period:
                st.json(item)
                for alert in item['alerts']:
                    st.warning(alert)
    st.caption('PSI и пороги покрытия — исследовательская диагностика. Они не заменяют согласованные SLA и регулярный мониторинг сервиса.')
    if (root/'data_audit.csv').exists():
        st.dataframe(pd.read_csv(root/'data_audit.csv'))
    if (root/'REPORT.md').exists():
        report=(root/'REPORT.md').read_text(encoding='utf-8')
        st.download_button('Скачать полный отчёт',report,file_name='REPORT.md')
        st.markdown(report)

```

</details>